# בניית אפליקציות ליצירת תמונות (Azure OpenAI)

יש הרבה יותר ב-LLMs מאשר יצירת טקסט. אפשר גם ליצור תמונות מתיאורים טקסטואליים. תמונות כמדיום שימושי בתחומי מדטק, אדריכלות, תיירות, פיתוח משחקים, שיווק ועוד. בשיעור הזה אנו עובדים עם מודלי **GPT Image** של היום ב-Microsoft Foundry.

## מטרות הלמידה

בסוף שיעור זה תוכל/י:

- להסביר מהי יצירת תמונה ואיפה היא שימושית.
- להבין את משפחת המודלים `gpt-image` וכיצד היא שונה ממודלי DALL·E הישנים.
- לבנות אפליקציה ליצירת תמונות ולערוך תמונות.

## מהי יצירת תמונה?

מודלים ליצירת תמונות מייצרים תמונות מתוך טקסט. מודלים מודרניים כמו `gpt-image` לומדים את הקשר בין טקסט ותמונות במהלך האימון, ואז הופכים רעש אקראי באופן איטרטיבי לתמונה שמתאימה לפקודת הטקסט.

שתי משפחות מפורסמות של מודלי תמונה הן:

- **`gpt-image` (OpenAI)** — הדור הנוכחי בו משתמשים בשיעור זה. תומך ביצירת תמונות מטקסט ועריכת תמונות (שחזור עם מסכה).
- **Midjourney** — מודל פופולרי של צד שלישי עם שירות משל עצמו וזרימת עבודה מבוססת Discord.

> מודלי התמונות הישנים של OpenAI — **DALL·E 2** ו-**DALL·E 3** — הם ישנים. DALL·E 3 כבר לא זמין לפריסות חדשות, ופיצ'רים כמו `create_variation` היו קיימים רק ב-DALL·E 2. יש להשתמש במודלים `gpt-image` לאפליקציות חדשות.

ב-Microsoft Foundry, **`gpt-image-2`** הוא מודל התמונה העדכני והיעיל ביותר ומומלץ כברירת מחדל. `gpt-image-1.5` ו-`gpt-image-1-mini` זמינים גם הם בדרך כלל.

> **חשוב:** מודלים `gpt-image` מחזירים את התמונה שנוצרה בפורמט **base64** (`b64_json`), לא ככתובת URL. הקוד שלך מפענח את מחרוזת הבסיס64 לבייטים ושומר אותה — אין URL של תמונה להורדה.


## בניית אפליקציית יצירת תמונות ראשונה שלך

אז מה נדרש לבניית אפליקציית יצירת תמונות? אתה צריך את הספריות הבאות:

- **python-dotenv**, מומלץ מאוד להשתמש בספרייה זו כדי לשמור את הסודות שלך בקובץ *.env* בנפרד מהקוד.
- **openai**, ספרייה זו היא שתשתמש בה כדי לתקשר עם ה-API של OpenAI.
- **pillow**, לעבודה עם תמונות בפייתון.

אם עדיין לא עשית זאת, עקוב אחר ההוראות בדף [Microsoft Learn](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/create-resource?pivots=web-portal&WT.mc_id=academic-105485-koreyst) ליצירת משאב ומודל ב-Microsoft Foundry. בחר **gpt-image-2** כמודל (המודל האחרון של Azure OpenAI ליצירת תמונות; DALL·E הוא מורשת).

1. צור קובץ *.env* עם התוכן הבא:

    ```text
    AZURE_OPENAI_ENDPOINT=<your endpoint>
    AZURE_OPENAI_API_KEY=<your key>
    AZURE_OPENAI_DEPLOYMENT="gpt-image-2"
    ```

    אתר מידע זה בפורטל Microsoft Foundry עבור המשאב שלך תחת מדור "Deployments".



1. אספו את הספריות הנ"ל בקובץ שנקרא *requirements.txt* כך:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. לאחר מכן, צרו סביבה וירטואלית והתקינו את הספריות:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> עבור Windows, השתמש בפקודות הבאות כדי ליצור ולהפעיל את סביבת העבודה הווירטואלית שלך:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. הוסף את הקוד הבא בקובץ בשם *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError

    # ייבא dotenv
    dotenv.load_dotenv()

    # תגדיר את לקוח Azure OpenAI (Microsoft Foundry).
    # דגמי תמונות זקוקים לגרסת API עדכנית - בדוק את תיעוד Foundry עבור הגרסה שהדגם שלך דורש.
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )

    try:
        # צור תמונה באמצעות ממשק ה-API ליצירת תמונות
        generation_response = client.images.generate(
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # הזן כאן את טקסט ההנחיה שלך
            size='1024x1024',
            n=1,
            model=os.environ['AZURE_OPENAI_DEPLOYMENT']   # למשל "gpt-image-2"
        )
        # הגדר את התיקייה לשמירת התמונה
        image_dir = os.path.join(os.curdir, 'images')

        # אם התיקייה לא קיימת, יצירתה
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # אתחל את נתיב התמונה (שים לב שסוג הקובץ צריך להיות png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # דגמי gpt-image מחזירים את התמונה כ-base64 (b64_json), לא כ-URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # הצג את התמונה בצופה התמונה המוגדר כברירת מחדל
        image = Image.open(image_path)
        image.show()

    # לטפל בחריגות
    except BadRequestError as err:
        print(err)

    ```

בואו נסביר את הקוד הזה:

- ראשית, אנחנו מייבאים את הספריות שאנו צריכים, כולל את ספריית OpenAI, הספרייה dotenv, המודול base64 וספריית Pillow.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError
    ```

- לאחר מכן, אנו טוענים את משתני הסביבה מקובץ *.env*.

    ```python
    # ייבא dotenv
    dotenv.load_dotenv()
    ```

- לאחר מכן, אנו מגדירים את לקוח Azure OpenAI (Microsoft Foundry).

    ```python
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )
    ```

- לאחר מכן, אנו מייצרים את התמונה:

    ```python
    generation_response = client.images.generate(
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # הקלד כאן את טקסט ההנחיה שלך
        size='1024x1024',
        n=1,
        model=os.environ['AZURE_OPENAI_DEPLOYMENT']
    )
    ```

    דגמי `gpt-image` מחזירים את התמונה כמחרוזת **base64** ב-`data[0].b64_json`. אנו מפענחים אותה לבתים וכותבים אותה לקובץ — אין URL להורדה.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- לבסוף, אנו פותחים את התמונה ומשתמשים בצופה התמונה הסטנדרטי כדי להציג אותה:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### פרטים נוספים על יצירת התמונה

בואו נבחן את הפרמטרים של `images.generate`:

- **prompt**, הוא הטקסט המשמש ליצירת התמונה. כאן זה "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils".
- **size**, הוא גודל התמונה המיוצרת (לדוגמה `1024x1024`, `1536x1024`, `1024x1536`, או `"auto"`).
- **n**, הוא מספר התמונות שנוצרו. כאן אנו יוצרים אחת.
- **model**, הוא שם הפרוסה של מודל התמונה שלך (לדוגמה `gpt-image-2`).

> דגמי תמונה אינם מקבלים פרמטר `temperature` — זהו בקרה ליצירת טקסט. לקבלת מגוון, קרא שוב ל-API; כדי לצמצם את המגוון, הפוך את ה-prompt שלך ליותר ספציפי.

## יכולות נוספות של יצירת תמונה

ראית איך ליצור תמונה עם כמה שורות Python. דגמי `gpt-image` יכולים גם **לערוך** תמונה קיימת. על ידי מתן תמונה, **מסכה** אופציונלית (שמסמנת את האזור לשינוי), ו-prompt, ניתן לשנות חלק מהתמונה — לדוגמה, להוסיף כובע לארנב שלנו.

```python
response = client.images.edit(
  model=os.environ['AZURE_OPENAI_DEPLOYMENT'],
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# גם העריכות מוחזרות כ-base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

תמונת הבסיס מכילה רק את הארנב; התמונה הסופית כוללת את הכובע על הארנב.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:
מסמך זה תורגם באמצעות שירות תרגום אוטומטי [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לקחת בחשבון שתרגומים אוטומטיים עלולים להכיל שגיאות או אי-דיוקים. יש להחשיב את המסמך המקורי בשפתו הטבעית כמקור הסמכות. למידע קריטי מומלץ להשתמש בתרגום מקצועי על ידי מתרגם אדם. אנו לא אחראים לכל אי-הבנה או פירוש שגוי הנובע מהשימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
